In [ ]:
import dotenv
dotenv.load_dotenv()
from dataclasses import dataclass
import glob
import os
from openai import OpenAI
from pymongo import MongoClient
import base64
import PyPDF2
import numpy as np

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_CORE_MODEL = os.getenv("OPENAI_CORE_MODEL")
OPENAI_EMBEDDEDING_MODEL = os.getenv("OPENAI_EMBEDDEDING_MODEL")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MONGODB_URI = os.getenv("MONGODB_URI")


## Make Connection

In [4]:
openai_client = OpenAI(api_key=OPENAI_API_KEY)
mongo_client = MongoClient(MONGODB_URI)
database = mongo_client['rag-data']
collection = database['vector-data']

# Files processing

In [5]:
@dataclass(frozen=True)
class TextData:
    content: str
    page: int | None

@dataclass(frozen=True)
class TextChunk:
    content: list
    vector: list
    page: int | None
    chunk_id: int

In [6]:
image_files = glob.glob("./input/*.png")
pdf_files = glob.glob("./input/*.pdf")


## Text extraction

In [7]:
def image_to_text_openai(file_path):
    with open(file_path, 'rb') as f:
        base64_image = base64.b64encode(f.read()).decode("utf-8")
    
    response = openai_client.responses.create(
        model=OPENAI_CORE_MODEL,
        input = [
            {
                'role': 'user',
                'content': [
                    {"type": "input_text", "text": "Extract text from this image"},
                    
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{base64_image}",
                    }
                ]
            }
        ]
    )
    return response

In [8]:
image_text = []
for image in image_files:
    extracted_image = image_to_text_openai(image)
    image_text.append(TextData(content = extracted_image.output_text, page = None))

In [9]:
pdf_text = []
for pdf in pdf_files:
    with open(pdf, 'rb') as f:
        read_pdf = PyPDF2.PdfReader(pdf)
        n_page = len(read_pdf.pages)
        
        for n in range(n_page):
            page = read_pdf.pages[n]
            page_text = page.extract_text()
        
            pdf_text.append(TextData(content = page_text, page = n+1))


# Chunking data

In [10]:
def chunks_text(text, chunk_size = 100, overlap = 20):
    chunks = []
    text_split = text.split()
    
    index = 0
    eof = len(text_split) - chunk_size + overlap
    print('index: ', index)
    print('eof: ', eof)

    if eof < 0:
        print(f'index {index} | {eof} eof')
        chunks.append(' '.join(text_split[0:]))
    else:
        while index < eof :
            if index == 0:
                chunk = text_split[index: chunk_size + overlap]
            else:
                chunk = text_split[index-overlap:index + chunk_size + overlap] # middle
            chunks.append(' '.join(chunk))
            index += chunk_size
            print(f'index {index} | {eof} eof')
        chunks.append(' '.join(text_split[index-overlap:]))

    print(f'Data contain {len(chunks)}')
    return chunks


# Embedded chunks

In [11]:
database = []

def embedding_chunks(texts):
    response = openai_client.embeddings.create(
        model= 'text-embedding-3-small',
        input= texts
    )
    return response

for page in pdf_text:
    chunks = chunks_text(page.content)
    for id, chunk in enumerate(chunks):
        database.append(TextChunk(content = chunk, vector = np.array(embedding_chunks(chunk).data[0].embedding, dtype=np.float32), page = page.page, chunk_id=id))

index:  0
eof:  279
index 100 | 279 eof
index 200 | 279 eof
index 300 | 279 eof
Data contain 4
index:  0
eof:  -4
index 0 | -4 eof
Data contain 1


# Similarity finding

In [12]:

def cosin_similarity(vector1, vector2):
    x = np.dot(vector1, vector2)
    y = (np.linalg.norm(vector1) * np.linalg.norm(vector2))
    if y == 0:
        return 0.0
    return float(x / y)

In [13]:
def retrieve(database, question, k = 3):
    results = []
    question_embedded = np.array(embedding_chunks(question).data[0].embedding, dtype=np.float32)
    for data in database:
        score = cosin_similarity(question_embedded, data.vector)
        results.append({
            'index': f'{data.page}_{data.chunk_id}',
            'page': data.page,
            'chunk_id': data.chunk_id,
            'content': data.content,
            'score': score
        })
    
    results.sort(key = lambda item: item['score'], reverse = True)
    return results[:k]

In [15]:
question = 'who am I'
x = retrieve(database, question, 8)

# Asking question

In [16]:
def gpt_requests(question):
    context_rag = retrieve(database, question, k = 5)
    response = openai_client.chat.completions.create(
        model = OPENAI_CORE_MODEL,
        messages = [
            {'role': 'system', 'content': "Answer the user question based on the provided context, if the context does not contain answer or no answer, say the data not found in the database. But if found, mention the index."},
            {"role": "user", "content": f'Question: {question}\n\n Data_Retrieval:\n {context_rag} The answer must return in json format as follows :  index:*Data_Retrieval index*, content:*Data_Retrieval content*, answer:*your answer*'}
        ]
    )

    return response

In [17]:
import json
answer = gpt_requests('Who is the punyawat and what does he do? give me the name and description')
gpt_answer = json.loads(answer.choices[0].message.content)
# gpt_answer


In [18]:
print(gpt_answer['answer'])

Name: Punyawat Jaroensiripong
Description: Punyawat Jaroensiripong is a Business Analyst, Cloud Architect, and AI Engineer. He worked at Mekha V Co., Ltd. (PTT Group) from August 2024 to August 2025, where he gathered business requirements across PTT internal groups to develop AI-driven solutions that improve operational efficiency and decision-making, and led a data science team to design and deploy Azure-based cloud solutions (including Azure OpenAI/foundry, Azure App Service, and Azure DevOps). He holds a Bachelor of Engineering in Computer Engineering from Mahidol University and is/was a Master of Information Science student (Computer Engineering) at Kyoto Institute of Technology. Key areas: machine learning/AI, image processing/segmentation, Azure cloud architecture, and security-related work.
